# Deep Learning - EEGNet-style LH/RH Classifier (Local CPU/GPU)

This notebook trains a PyTorch deep-learning model on wide EEG features.
Outputs are saved to results_local_cpu_gpu/deep_learning and saved_models.
Default mode is smoke for quick local validation.

In [2]:
import os
import random
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, cohen_kappa_score,
    classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_curve
)

HAS_TORCH = True
TORCH_IMPORT_ERROR = None
try:
    import torch
    import torch.nn as nn
    from torch.utils.data import Dataset, DataLoader
except Exception as e:
    HAS_TORCH = False
    TORCH_IMPORT_ERROR = str(e)

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

BASE_DIR = Path("/home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/classifier_notebook")
INPUT_PRIMARY = BASE_DIR / "results_local_cpu_gpu" / "preprocessing" / "features_lh_rh.csv"
INPUT_FALLBACK = BASE_DIR / "features_lh_rh.csv"
INPUT_CSV = INPUT_PRIMARY if INPUT_PRIMARY.exists() else INPUT_FALLBACK
RESULTS_DIR = BASE_DIR / "results_local_cpu_gpu" / "deep_learning"
MODEL_DIR = BASE_DIR / "results_local_cpu_gpu" / "saved_models"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RUN_MODE = "smoke"  # change to "full" for full training
RANDOM_STATE = 42
SMOKE_MAX_PER_CLASS = 1200

EPOCHS = 3 if RUN_MODE == "smoke" else 40
BATCH_SIZE = 64 if RUN_MODE == "smoke" else 128
LEARNING_RATE = 1e-3
PATIENCE = 2 if RUN_MODE == "smoke" else 8
WEIGHT_DECAY = 1e-4

def savefig(name: str):
    plt.tight_layout()
    path = RESULTS_DIR / name
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.close()

print(f"Input CSV: {INPUT_CSV}")
print(f"Output dir: {RESULTS_DIR}")
if not INPUT_CSV.exists():
    raise FileNotFoundError("features_lh_rh.csv not found. Run preprocessing notebook first.")

if not HAS_TORCH:
    print("PyTorch is not available in this environment.")
    print(f"Import error: {TORCH_IMPORT_ERROR}")
    print("Install torch to run deep learning training. Notebook setup is complete.")
else:
    random.seed(RANDOM_STATE)
    np.random.seed(RANDOM_STATE)
    torch.manual_seed(RANDOM_STATE)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_STATE)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if torch.cuda.is_available():
        print(f"CUDA device: {torch.cuda.get_device_name(0)}")

    df = pd.read_csv(INPUT_CSV)
    if RUN_MODE == "smoke":
        sampled = []
        for _, part in df.groupby("label"):
            take_n = min(len(part), SMOKE_MAX_PER_CLASS)
            sampled.append(part.sample(take_n, random_state=RANDOM_STATE))
        df = pd.concat(sampled, axis=0).sample(frac=1.0, random_state=RANDOM_STATE).reset_index(drop=True)

    meta_cols = {"subject_id", "scenario_id", "scenario", "filename", "task", "label", "label_name"}
    feature_cols = [c for c in df.columns if c not in meta_cols]

    X = df[feature_cols].apply(pd.to_numeric, errors="coerce").fillna(0.0).values.astype(np.float32)
    y = df["label"].astype(int).values

    print(f"Dataset: X={X.shape}, y={y.shape}")
    print(df["label_name"].value_counts())

    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
    )
    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval, test_size=0.2, stratify=y_trainval, random_state=RANDOM_STATE
    )

    scaler = RobustScaler()
    X_train = scaler.fit_transform(X_train).astype(np.float32)
    X_val = scaler.transform(X_val).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    class EEGDataset(Dataset):
        def __init__(self, X_arr, y_arr):
            self.X = torch.from_numpy(X_arr).float()
            self.y = torch.from_numpy(y_arr).long()

        def __len__(self):
            return len(self.y)

        def __getitem__(self, idx):
            return self.X[idx], self.y[idx]

    train_loader = DataLoader(EEGDataset(X_train, y_train), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
    val_loader = DataLoader(EEGDataset(X_val, y_val), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    test_loader = DataLoader(EEGDataset(X_test, y_test), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    class TabularEEGNet(nn.Module):
        def __init__(self, input_dim: int, dropout: float = 0.35):
            super().__init__()
            self.block1 = nn.Sequential(
                nn.Conv1d(1, 16, kernel_size=7, padding=3, bias=False),
                nn.BatchNorm1d(16),
                nn.ELU(),
            )
            self.block2 = nn.Sequential(
                nn.Conv1d(16, 16, kernel_size=5, padding=2, groups=16, bias=False),
                nn.Conv1d(16, 32, kernel_size=1, bias=False),
                nn.BatchNorm1d(32),
                nn.ELU(),
            )
            self.pool = nn.AdaptiveAvgPool1d(1)
            self.drop = nn.Dropout(dropout)
            self.fc = nn.Linear(32, 2)

        def forward(self, x):
            x = x.unsqueeze(1)
            x = self.block1(x)
            x = self.block2(x)
            x = self.pool(x).squeeze(-1)
            x = self.drop(x)
            return self.fc(x)

    model = TabularEEGNet(input_dim=X_train.shape[1]).to(device)

    class_counts = np.bincount(y_train)
    class_weights = torch.tensor([1.0 / max(class_counts[0], 1), 1.0 / max(class_counts[1], 1)], dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    def run_epoch(loader, training=False):
        if training:
            model.train()
        else:
            model.eval()

        total_loss = 0.0
        all_probs = []
        all_preds = []
        all_true = []

        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)

            if training:
                optimizer.zero_grad()

            with torch.set_grad_enabled(training):
                logits = model(xb)
                loss = criterion(logits, yb)
                probs = torch.softmax(logits, dim=1)[:, 1]
                preds = torch.argmax(logits, dim=1)

                if training:
                    loss.backward()
                    optimizer.step()

            total_loss += loss.item() * len(yb)
            all_probs.append(probs.detach().cpu().numpy())
            all_preds.append(preds.detach().cpu().numpy())
            all_true.append(yb.detach().cpu().numpy())

        y_true = np.concatenate(all_true)
        y_pred = np.concatenate(all_preds)
        y_prob = np.concatenate(all_probs)

        loss_mean = total_loss / max(len(y_true), 1)
        acc = accuracy_score(y_true, y_pred)
        bac = balanced_accuracy_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan

        return {
            "loss": loss_mean,
            "acc": acc,
            "bac": bac,
            "f1": f1,
            "auc": auc,
            "y_true": y_true,
            "y_pred": y_pred,
            "y_prob": y_prob,
        }

    history = []
    best_state = None
    best_auc = -np.inf
    best_epoch = 0

    for epoch in range(1, EPOCHS + 1):
        train_res = run_epoch(train_loader, training=True)
        val_res = run_epoch(val_loader, training=False)

        history.append({
            "epoch": epoch,
            "train_loss": train_res["loss"],
            "val_loss": val_res["loss"],
            "train_acc": train_res["acc"],
            "val_acc": val_res["acc"],
            "val_auc": val_res["auc"],
        })

        val_auc = -np.inf if np.isnan(val_res["auc"]) else val_res["auc"]
        if val_auc > best_auc:
            best_auc = val_auc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"train_loss={train_res['loss']:.4f}, val_loss={val_res['loss']:.4f}, "
            f"train_acc={train_res['acc']:.4f}, val_acc={val_res['acc']:.4f}, val_auc={val_res['auc']:.4f}"
        )

        if epoch - best_epoch >= PATIENCE:
            print("Early stopping triggered.")
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    test_res = run_epoch(test_loader, training=False)
    print("\nTest metrics:")
    print(f"Accuracy: {test_res['acc']:.4f}")
    print(f"Balanced Accuracy: {test_res['bac']:.4f}")
    print(f"F1: {test_res['f1']:.4f}")
    print(f"ROC-AUC: {test_res['auc']:.4f}")
    print(f"Cohen Kappa: {cohen_kappa_score(test_res['y_true'], test_res['y_pred']):.4f}")

    print("\nClassification report:")
    print(classification_report(test_res['y_true'], test_res['y_pred'], target_names=["LH", "RH"], zero_division=0))

    history_df = pd.DataFrame(history)
    history_df.to_csv(RESULTS_DIR / "training_history.csv", index=False)

    metrics_df = pd.DataFrame([
        {
            "Model": "TabularEEGNet",
            "Accuracy": test_res['acc'],
            "Balanced_Acc": test_res['bac'],
            "F1": test_res['f1'],
            "ROC_AUC": test_res['auc'],
            "Cohen_Kappa": cohen_kappa_score(test_res['y_true'], test_res['y_pred']),
            "Run_Mode": RUN_MODE,
            "Device": str(device),
            "Best_Epoch": best_epoch,
        }
    ])
    metrics_df.to_csv(RESULTS_DIR / "deep_learning_metrics.csv", index=False)

    plt.figure(figsize=(10, 4))
    plt.plot(history_df['epoch'], history_df['train_loss'], label='Train Loss')
    plt.plot(history_df['epoch'], history_df['val_loss'], label='Val Loss')
    plt.title('Training Curves - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    savefig("training_curves_loss.png")

    plt.figure(figsize=(10, 4))
    plt.plot(history_df['epoch'], history_df['train_acc'], label='Train Accuracy')
    plt.plot(history_df['epoch'], history_df['val_acc'], label='Val Accuracy')
    if history_df['val_auc'].notna().any():
        plt.plot(history_df['epoch'], history_df['val_auc'], label='Val AUC')
    plt.title('Training Curves - Accuracy/AUC')
    plt.xlabel('Epoch')
    plt.ylabel('Score')
    plt.legend()
    savefig("training_curves_scores.png")

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    cm = confusion_matrix(test_res['y_true'], test_res['y_pred'])
    ConfusionMatrixDisplay(cm, display_labels=["LH", "RH"]).plot(ax=axes[0], colorbar=False)
    axes[0].set_title('Confusion Matrix (Counts)')

    cm_norm = confusion_matrix(test_res['y_true'], test_res['y_pred'], normalize='true')
    ConfusionMatrixDisplay(cm_norm, display_labels=["LH", "RH"]).plot(ax=axes[1], colorbar=False)
    axes[1].set_title('Confusion Matrix (Normalized)')
    savefig("confusion_matrix_dl.png")

    fpr, tpr, _ = roc_curve(test_res['y_true'], test_res['y_prob'])
    plt.figure(figsize=(6, 6))
    plt.plot(fpr, tpr, label=f"TabularEEGNet (AUC={test_res['auc']:.3f})")
    plt.plot([0, 1], [0, 1], 'k--', label='Random')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve - Deep Learning')
    plt.legend(loc='lower right')
    savefig("roc_curve_dl.png")

    model_path = MODEL_DIR / "tabular_eegnet_best.pt"
    scaler_path = MODEL_DIR / "tabular_eegnet_scaler.joblib"
    feature_path = MODEL_DIR / "tabular_eegnet_feature_cols.txt"

    torch.save({
        "model_state_dict": model.state_dict(),
        "input_dim": X_train.shape[1],
        "run_mode": RUN_MODE,
        "best_epoch": best_epoch,
    }, model_path)
    joblib.dump(scaler, scaler_path)
    with open(feature_path, "w", encoding="utf-8") as f:
        for col in feature_cols:
            f.write(col + "\n")

    print("Saved deep learning files:")
    for p in sorted(RESULTS_DIR.glob("*")):
        if p.is_file():
            print(f"- {p.name} ({p.stat().st_size / 1024:.1f} KB)")
    print("Saved model files:")
    for p in sorted(MODEL_DIR.glob("tabular_eegnet*")):
        if p.is_file():
            print(f"- {p.name} ({p.stat().st_size / 1024:.1f} KB)")

Input CSV: /home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/classifier_notebook/results_local_cpu_gpu/preprocessing/features_lh_rh.csv
Output dir: /home/jeremy-mboe/Documents/Kuliah/Sem4/EEG_ALS/WEB/classifier_notebook/results_local_cpu_gpu/deep_learning
Device: cuda
CUDA device: NVIDIA GeForce RTX 4060 Laptop GPU
Dataset: X=(300, 27), y=(300,)
label_name
RH    150
LH    150
Name: count, dtype: int64
Epoch 01/3 | train_loss=0.7212, val_loss=0.7061, train_acc=0.4844, val_acc=0.5000, val_auc=0.3767
Epoch 02/3 | train_loss=0.7253, val_loss=0.7049, train_acc=0.4635, val_acc=0.5000, val_auc=0.3663
Epoch 03/3 | train_loss=0.6903, val_loss=0.7039, train_acc=0.5365, val_acc=0.5000, val_auc=0.3628
Early stopping triggered.

Test metrics:
Accuracy: 0.5000
Balanced Accuracy: 0.5000
F1: 0.6667
ROC-AUC: 0.4322
Cohen Kappa: 0.0000

Classification report:
              precision    recall  f1-score   support

          LH       0.00      0.00      0.00        30
          RH       0.50      1.00  